CREATE THE CP

Run in conda env nonconformist

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from nonconformist.base import ClassifierAdapter
from nonconformist.cp import IcpClassifier
from nonconformist.nc import ClassifierNc, MarginErrFunc
from rdkit.Chem import AllChem
from rdkit.Chem import Mol, MolFromSmiles
from rdkit.DataStructs.cDataStructs import UIntSparseIntVect

from typing import List, Tuple
import sys


def smiles_to_fingerprints(
    smiles_list: List[str],
    radius: int = 3,
    n_bits: int = 2048,
    use_features: bool = False,
):
    """Return a list of RDKit ExplicitBitVect fingerprint objects."""
    mols = [MolFromSmiles(s) for s in smiles_list]
    mols = [m for m in mols if m is not None]

    fps = [
        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius,
            nBits=n_bits,
            useFeatures=use_features,
        )
        for mol in mols
    ]
    return fps


# Load your dataset
# df has columns: [fingerprint features...] + 'activity' (continuous, e.g. pIC50)
df = pd.read_csv('../00_data/egfr/cleaned_egfr.csv')
df["fp"]= smiles_to_fingerprints(df["canonical_smiles"])
fps_matrix = np.stack([np.array(fp) for fp in df["fp"]])



In [ ]:
# Set activity threshold used to binarize labels (0=inactive, 1=active)
THRESHOLD = 6.0

# Alternative stricter threshold for harder classification
#THRESHOLD = 8.5

df['label'] = (df['pChEMBL Value'] >= THRESHOLD).astype(int)  # 0=inactive, 1=active

# Build train/calibration/test splits from pre-defined split column
X_train = fps_matrix[df["split"]=="train"]
y_train = df[df["split"]=="train"]['label'].values

X_cal= fps_matrix[df["split"]=="val"]
y_cal = df[df["split"]=="val"]['label'].values

X_test = fps_matrix[df["split"]=="test"]
y_test = df[df["split"]=="test"]['label'].values

In [ ]:
# Base model - any sklearn classifier would work
base_model = RandomForestClassifier(n_estimators=500, random_state=42)
model_adapter = ClassifierAdapter(base_model)

nc = ClassifierNc(model_adapter, err_func=MarginErrFunc())

icp = IcpClassifier(nc)

# Fit on train set only
icp.fit(X_train, y_train)

icp.calibrate(X_cal, y_cal)

In [ ]:
# significance level: 0.05 = 95% coverage guarantee
SIGNIFICANCE = 0.05

# Returns a boolean array: shape (n_samples, n_classes)
# True means that class is IN the prediction set
prediction_sets = icp.predict(X_test, significance=SIGNIFICANCE)

# Decode into human-readable output
results = []
for i, ps in enumerate(prediction_sets):
    included = [('inactive', 'active')[j] for j, v in enumerate(ps) if v]
    set_size = len(included)

    if set_size == 1:
        certainty = 'confident'
    elif set_size == 2:
        certainty = 'uncertain'   # ← the "middle class"
    else:
        certainty = 'anomalous'

    results.append({
        'compound_idx': i,
        'prediction_set': included,
        'set_size': set_size,
        'certainty': certainty
    })

results_df = pd.DataFrame(results)
print(results_df.head(20))

    compound_idx      prediction_set  set_size  certainty
0              0  [inactive, active]         2  uncertain
1              1  [inactive, active]         2  uncertain
2              2  [inactive, active]         2  uncertain
3              3  [inactive, active]         2  uncertain
4              4  [inactive, active]         2  uncertain
5              5  [inactive, active]         2  uncertain
6              6  [inactive, active]         2  uncertain
7              7  [inactive, active]         2  uncertain
8              8  [inactive, active]         2  uncertain
9              9  [inactive, active]         2  uncertain
10            10  [inactive, active]         2  uncertain
11            11  [inactive, active]         2  uncertain
12            12  [inactive, active]         2  uncertain
13            13  [inactive, active]         2  uncertain
14            14  [inactive, active]         2  uncertain
15            15  [inactive, active]         2  uncertain
16            

In [6]:
# Verify empirical error without nonconformist.evaluation (it depends on old sklearn APIs)
prediction_sets = icp.predict(X_test, significance=SIGNIFICANCE).astype(bool)

# Error = true class not included in the conformal prediction set
y_test_int = y_test.astype(int)
errors = np.mean(~prediction_sets[np.arange(len(y_test_int)), y_test_int])

print(f"Empirical error rate: {errors:.3f}  (target: {SIGNIFICANCE})")
# Should be close to or below the target when calibration is adequate

Empirical error rate: 0.000  (target: 0.0005)


In [7]:
# Get raw p-values for each class
p_vals = icp.predict(X_test, significance=None)  # shape: (n, 2)

results_df['p_inactive'] = p_vals[:, 0]
results_df['p_active']   = p_vals[:, 1]

# Uncertainty = how close the two p-values are (small gap = uncertain)
results_df['uncertainty'] = 1 - abs(
    results_df['p_active'] - results_df['p_inactive']
)

In [ ]:
# Quick distribution check for target potency values
df["pChEMBL Value"].describe()

count    45275.000000
mean         6.945539
std          1.274057
min          2.050000
25%          6.035000
50%          7.000000
75%          7.865000
max         11.000000
Name: pChEMBL Value, dtype: float64

In [ ]:
# Display fitted ICP object summary
icp

,nc_function,ClassifierNc(...om_state=42)))
,condition,<function Bas...x7f5b7576c820>
,smoothing,True
,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None


In [ ]:
from pathlib import Path
import joblib

# cloudpickle can serialize local lambdas inside nonconformist objects
try:
    import cloudpickle
except ImportError as e:
    raise ImportError("Install cloudpickle first: pip install cloudpickle") from e

# Save paths
save_path = Path('./egfrFull_conformal.pkl')
fallback_path = Path('./egfrFull_portable.joblib')
save_path.parent.mkdir(parents=True, exist_ok=True)

# Full artifact (preferred): includes fitted ICP object
artifact_full = {
    'icp': icp,
    'threshold': THRESHOLD,
    'significance': SIGNIFICANCE,
    'n_bits': 2048,
    'class_names': ['inactive', 'active']
}

try:
    with open(save_path, 'wb') as f:
        cloudpickle.dump(artifact_full, f)

    print(f'Saved full conformal model to: {save_path}')
except Exception as e:
    print(f'Full ICP save failed: {e}')
    print('Saving portable fallback (without icp) ...')

    # Fallback artifact: usable for probability-based uncertainty in RL
    artifact_fallback = {
        'rf_model': model_adapter.model,  # fitted sklearn RF
        'threshold': THRESHOLD,
        'significance': SIGNIFICANCE,
        'n_bits': 2048,
        'class_names': ['inactive', 'active']
    }
    joblib.dump(artifact_fallback, fallback_path, compress=3)
    print(f'Saved fallback artifact to: {fallback_path}')


In [ ]:
# Load saved artifact and restore ICP object
import cloudpickle

with open("./egfrFull_conformal.pkl", 'rb') as f:
    loaded = cloudpickle.load(f)
    
print('Loaded keys:', loaded.keys())
icp=loaded["icp"]

Loaded keys: dict_keys(['icp', 'threshold', 'significance', 'n_bits', 'class_names'])
